# Optimal SNR: GWSNR versus Bilby

Compare the **actual GWSNR `inner_product` API** with Bilby's detector-response optimal SNR for `IMRPhenomD` and `IMRPhenomXPHM`, using the event in `pe_compare_image_1.py`.

This is a deterministic, zero-noise comparison of optimal SNR, not a noise-dependent matched-filter SNR. The original script computes a GWSNR-like expression without calling GWSNR; here we test both the API and that expression.

Run all cells with the **ler (Python 3.10)** kernel. The notebook reports the imported package paths, verifies the settings, and retains the numerical results. This notebook verifies the corrected local implementation, including the mixed-polarization terms. Restart the kernel and run all cells after editing the package; an editable install does not reload modules already imported in a running kernel.

In [1]:
from pathlib import Path
import sys
from importlib.metadata import version

import numpy as np
import pandas as pd
from IPython.display import display
import bilby
import gwsnr
from gwsnr import GWSNR
from gwsnr.numba import antenna_response_array

# Locate this checkout whether Jupyter starts in its root or examples/ directory.
REPO_ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "gwsnr/core/gwsnr.py").is_file()
)
assert Path(gwsnr.__file__).resolve() == REPO_ROOT / "gwsnr/__init__.py", (
    "The kernel imported a different GWSNR checkout. Restart with the ler kernel."
)
print(f"Python:   {sys.version.split()[0]} ({sys.executable})")
print(f"GWSNR:    {gwsnr.__version__} ({gwsnr.__file__})")
print(f"Bilby:    {bilby.__version__} ({bilby.__file__})")
print(f"LALSuite: {version('lalsuite')}")

Python:   3.10.18 (/Users/phurailatpamhemantakumar/anaconda3/envs/ler/bin/python)
GWSNR:    0.5.2 (/Users/phurailatpamhemantakumar/phd/mypackages/gwsnr/gwsnr/__init__.py)
Bilby:    2.5.2 (/Users/phurailatpamhemantakumar/anaconda3/envs/ler/lib/python3.10/site-packages/bilby/__init__.py)
LALSuite: 7.26.1


## 1. Shared settings and source parameters

Masses are detector-frame solar masses, distance is in Mpc, angles are in radians, and `geocent_time` is GPS seconds. Magnification is 1, as in the supplied script, so the distance is unchanged.

Both calculations use a 64 s waveform, sampling at 1024 Hz, a 20–512 Hz analysis band, and a reference frequency of 20 Hz. Thus $\Delta f=1/64$ Hz. The files below contain **ASD**, which each library squares to obtain PSD; the sampled PSD arrays are checked explicitly.

In [2]:
DURATION = 64.0
SAMPLING_FREQUENCY = 1024.0
F_MIN = 20.0
F_MAX = SAMPLING_FREQUENCY / 2.0
F_REF = 20.0
DETECTORS = ["L1", "H1", "V1"]
ASD_FILES = {
    "L1": "aLIGO_O4_high_asd.txt",
    "H1": "aLIGO_O4_high_asd.txt",
    "V1": "AdV_asd.txt",
}

# API comparison allows the small antenna-response approximation difference.
# Reconstruction checks below use a stricter relative tolerance of 1e-12.
SNR_RTOL = 1e-7
SNR_ATOL = 1e-10

injection_parameters = {
    "mass_1": 39.39697565794667,
    "mass_2": 38.502200779605765,
    "a_1": 0.013933539741033549,
    "a_2": 0.10387844924090532,
    "tilt_1": 0.3683219347284615,
    "tilt_2": 1.2773472444692886,
    "phi_12": 5.512838537731936,
    "phi_jl": 0.2848074437530412,
    "luminosity_distance": 696.8177279507435,
    "theta_jn": 0.8278270206864476,
    "psi": 2.583807400988801,
    "phase": 0.09150141867219445,
    "geocent_time": 1252983658.1262412,
    "ra": 5.816668997978307,
    "dec": 0.179951954224289,
}

### Model-specific spin inputs

`IMRPhenomXPHM` uses the original injection unchanged. `IMRPhenomD` is nonprecessing: LAL rejects the original nonzero transverse spins. Its control case sets both tilts and the spin azimuths to zero, retaining the supplied spin magnitudes and all other parameters. This is a separate aligned-spin source, not a claim that the two waveform models describe the same source.

Within each model, **the same parameter dictionary is passed to both libraries**.

In [3]:
parameters_by_model = {
    "IMRPhenomD": {
        **injection_parameters,
        "tilt_1": 0.0,
        "tilt_2": 0.0,
        "phi_12": 0.0,
        "phi_jl": 0.0,
    },
    "IMRPhenomXPHM": injection_parameters.copy(),
}
display(pd.DataFrame(parameters_by_model).rename_axis("parameter"))

,IMRPhenomD,IMRPhenomXPHM
parameter,,
mass_1,3.939698e+01,3.939698e+01
mass_2,3.850220e+01,3.850220e+01
a_1,1.393354e-02,1.393354e-02
a_2,1.038784e-01,1.038784e-01
tilt_1,0.000000e+00,3.683219e-01
tilt_2,0.000000e+00,1.277347e+00
phi_12,0.000000e+00,5.512839e+00
phi_jl,0.000000e+00,2.848074e-01
luminosity_distance,6.968177e+02,6.968177e+02


## 2. Compare the actual APIs and audit the settings

For each model we independently generate the waveform through GWSNR and through Bilby. We check the duration, sampling and reference frequencies, source model, detector order/tensors, frequency grid, sampled PSDs, and antenna responses. `fixed_duration=64` disables GWSNR's automatic duration estimate; mass cuts and SNR recalculation are disabled explicitly.

Bilby's zero-noise strain data have unit window factor and identity calibration. Its segment starts at $t_c-T/2$, while both waveform generators start at zero. The detector arrival-time shift is a common unit-modulus factor in frequency space and cancels in the optimal SNR. The checks below also verify this via the projected-strain norm.

In [4]:
def make_bilby_setup(approximant, parameters):
    """Construct Bilby's waveform generator and zero-noise detector network."""
    ifos = bilby.gw.detector.InterferometerList(DETECTORS)
    for ifo in ifos:
        ifo.power_spectral_density = bilby.gw.detector.PowerSpectralDensity(
            asd_file=ASD_FILES[ifo.name]
        )
        ifo.minimum_frequency = F_MIN
        ifo.maximum_frequency = F_MAX
        ifo.set_strain_data_from_zero_noise(
            sampling_frequency=SAMPLING_FREQUENCY,
            duration=DURATION,
            start_time=parameters["geocent_time"] - DURATION / 2,
        )

    generator = bilby.gw.WaveformGenerator(
        duration=DURATION,
        sampling_frequency=SAMPLING_FREQUENCY,
        start_time=0.0,
        frequency_domain_source_model=bilby.gw.source.lal_binary_black_hole,
        parameter_conversion=bilby.gw.conversion.convert_to_lal_binary_black_hole_parameters,
        waveform_arguments={
            "waveform_approximant": approximant,
            "reference_frequency": F_REF,
            "minimum_frequency": F_MIN,
            "maximum_frequency": F_MAX,
        },
    )
    return ifos, generator


def make_gwsnr(approximant):
    """Use the LAL inner-product backend, not interpolation or an ANN."""
    return GWSNR(
        npool=1,
        snr_method="inner_product",
        snr_type="optimal_snr",
        waveform_approximant=approximant,
        frequency_domain_source_model="lal_binary_black_hole",
        sampling_frequency=SAMPLING_FREQUENCY,
        minimum_frequency=F_MIN,
        reference_frequency=F_REF,
        fixed_duration=DURATION,
        duration_min=DURATION,
        duration_max=DURATION,
        psds=ASD_FILES.copy(),
        ifos=DETECTORS.copy(),
        mtot_cut=False,
        snr_recalculation=False,
        # Reuse the existing directory; no interpolator is needed for this test.
        interpolator_dir=str(REPO_ROOT / "examples/interpolator_json"),
        gwsnr_verbose=False,
        multiprocessing_verbose=False,
    )

## 3. Identify any difference in the inner product

With the complex noise-weighted product
\[
I(a,b)=4\Delta f\sum_{f\in[20,512]\,\mathrm{Hz}}\frac{a^*(f)b(f)}{S_n(f)},
\]
the full detector norm is
\[
\rho_{\rm full}^2=F_+^2 I(h_+,h_+)+F_\times^2 I(h_\times,h_\times)
+\underbrace{2F_+F_\times\operatorname{Re}I(h_+,h_\times)}_{C}.
\]

The corrected GWSNR implementation computes the complex mixed product $I(h_+,h_\times)$ and includes both conjugate terms through $2F_+F_\times\operatorname{Re}I(h_+,h_\times)$. For the aligned-spin, dominant-mode `IMRPhenomD` waveform this contribution vanishes up to numerical precision; it need not vanish for `IMRPhenomXPHM`.

The diagnostic verifies the actual API against the full projected-strain norm. `gwsnr_diagonal_only` reconstructs the former result for comparison; it is not the corrected API output.

GWSNR uses a linear GPS-to-GMST approximation (`gwsnr/numba/njit_functions.py`, `gps_to_gmst`), so its antenna factors differ slightly from Bilby's. We compute the full power with each set of antenna factors, define their difference as $A=\rho^2_{\rm full,Bilby\ factors}-\rho^2_{\rm full,GWSNR\ factors}$, and verify $\rho_{\rm Bilby}^2-\rho_{\rm GWSNR}^2=A$. Both norms now include the mixed terms. No post-hoc correction is applied to either API output.

In [5]:
def compare_model(approximant, parameters):
    ifos, generator = make_bilby_setup(approximant, parameters)
    calculator = make_gwsnr(approximant)

    assert calculator.snr_method == "inner_product"
    assert calculator.snr_type == "optimal_snr"
    assert calculator.waveform_approximant == approximant
    assert calculator.frequency_domain_source_model == "lal_binary_black_hole"
    assert calculator.fixed_duration == generator.duration == DURATION
    assert calculator.sampling_frequency == generator.sampling_frequency == SAMPLING_FREQUENCY
    assert calculator.f_min == F_MIN and calculator.f_ref == F_REF
    assert calculator.detector_list == DETECTORS
    assert not calculator.mtot_cut and not calculator.snr_recalculation

    # These are the actual GWSNR API results, with no post-hoc correction.
    api_snrs = calculator.optimal_snr_with_inner_product(
        gw_param_dict=parameters.copy(), output_jsonfile=False
    )
    polarizations = generator.frequency_domain_strain(parameters.copy())
    frequencies = generator.frequency_array
    np.testing.assert_allclose(np.diff(frequencies), 1 / DURATION, rtol=0, atol=1e-14)
    assert frequencies[0] == 0.0 and frequencies[-1] == F_MAX
    band = (frequencies >= F_MIN) & (frequencies <= F_MAX)
    hp, hc = polarizations["plus"], polarizations["cross"]
    assert np.all(np.isfinite(hp)) and np.all(np.isfinite(hc))
    assert np.any(np.abs(hp[band]) > 0)
    # GWSNR's worker uses all valid PSD bins: out-of-band waveform bins must be zero.
    assert np.all(hp[~band] == 0) and np.all(hc[~band] == 0)

    fp_gw, fc_gw = antenna_response_array(
        *[np.array([parameters[key]]) for key in ("ra", "dec", "geocent_time", "psi")],
        np.array(calculator.detector_tensor_list),
    )
    rows, audits = [], []
    for index, ifo in enumerate(ifos):
        np.testing.assert_array_equal(ifo.frequency_array, frequencies)
        np.testing.assert_array_equal(ifo.frequency_mask, band)
        np.testing.assert_array_equal(calculator.detector_tensor_list[index], ifo.detector_tensor)
        assert ifo.strain_data.window_factor == 1.0
        calibration = ifo.calibration_model.get_calibration_factor(
            frequencies[band], prefix=f"recalib_{ifo.name}_", **parameters
        )
        np.testing.assert_allclose(calibration, 1.0, rtol=0, atol=0)

        psd_bilby = ifo.power_spectral_density_array
        psd_gwsnr = calculator.psds_list[index][2](frequencies)
        np.testing.assert_allclose(psd_gwsnr, psd_bilby, rtol=1e-13, atol=0)
        assert np.all(np.isfinite(psd_bilby[band])) and np.all(psd_bilby[band] > 0)

        fp, fc = [ifo.antenna_response(
            parameters["ra"], parameters["dec"], parameters["geocent_time"],
            parameters["psi"], mode=mode,
        ) for mode in ("plus", "cross")]
        np.testing.assert_allclose(
            [fp_gw[index, 0], fc_gw[index, 0]], [fp, fc], rtol=0, atol=1e-7
        )

        def inner(a, b):
            return 4 / DURATION * np.sum(np.conj(a[band]) * b[band] / psd_bilby[band])

        diagonal = float(np.real(fp**2 * inner(hp, hp) + fc**2 * inner(hc, hc)))
        diagonal_gwsnr = float(np.real(
            fp_gw[index, 0]**2 * inner(hp, hp) + fc_gw[index, 0]**2 * inner(hc, hc)
        ))
        cross_term = float(2 * fp * fc * np.real(inner(hp, hc)))
        cross_term_gwsnr = float(2 * fp_gw[index, 0] * fc_gw[index, 0] * np.real(inner(hp, hc)))
        full_gwsnr = diagonal_gwsnr + cross_term_gwsnr
        projected = fp * hp + fc * hc
        projected_rho2 = float(np.real(inner(projected, projected)))
        antenna_term = projected_rho2 - full_gwsnr
        strain = ifo.get_detector_response(polarizations, parameters)
        bilby_rho2 = float(np.real(ifo.optimal_snr_squared(strain)))
        gwsnr_snr = float(api_snrs[f"optimal_snr_{ifo.name}"][0])

        np.testing.assert_allclose(projected_rho2, diagonal + cross_term, rtol=1e-12)
        np.testing.assert_allclose(bilby_rho2, projected_rho2, rtol=1e-12)
        np.testing.assert_allclose(gwsnr_snr, np.sqrt(full_gwsnr), rtol=1e-12, atol=1e-12)
        # Scale by rho^2 because the residual itself should vanish.
        residual = bilby_rho2 - gwsnr_snr**2 - antenna_term
        assert abs(residual) <= 1e-12 * bilby_rho2
        rows.append({
            "model": approximant, "detector": ifo.name,
            "gwsnr": gwsnr_snr, "bilby": np.sqrt(bilby_rho2),
            "gwsnr_diagonal_only": np.sqrt(diagonal_gwsnr),
            "diagonal_rho2": diagonal, "cross_term_rho2": cross_term,
            "antenna_term_rho2": antenna_term,
            "rho2_residual": residual,
        })
        audits.append({
            "model": approximant, "detector": ifo.name,
            "ASD_file": str(ifo.power_spectral_density.asd_file),
            "max_PSD_relative_difference": float(np.max(np.abs(psd_gwsnr[band] / psd_bilby[band] - 1))),
            "Fp_difference": fp_gw[index, 0] - fp,
            "Fc_difference": fc_gw[index, 0] - fc,
        })

    network_gwsnr = float(api_snrs["optimal_snr_net"][0])
    network_bilby = np.sqrt(sum(row["bilby"]**2 for row in rows))
    np.testing.assert_allclose(network_gwsnr, np.sqrt(sum(row["gwsnr"]**2 for row in rows)), rtol=1e-13)
    rows.append({
        "model": approximant, "detector": "network",
        "gwsnr": network_gwsnr, "bilby": network_bilby,
        "gwsnr_diagonal_only": np.sqrt(sum(row["gwsnr_diagonal_only"]**2 for row in rows)),
        "diagonal_rho2": sum(row["diagonal_rho2"] for row in rows),
        "cross_term_rho2": sum(row["cross_term_rho2"] for row in rows),
        "antenna_term_rho2": sum(row["antenna_term_rho2"] for row in rows),
        "rho2_residual": sum(row["rho2_residual"] for row in rows),
    })
    return rows, audits

In [6]:
all_rows, all_audits = [], []
for approximant, parameters in parameters_by_model.items():
    rows, audits = compare_model(approximant, parameters)
    all_rows.extend(rows)
    all_audits.extend(audits)

results = pd.DataFrame(all_rows)
results["difference"] = results["gwsnr"] - results["bilby"]
results["difference_percent"] = 100 * results["difference"] / results["bilby"]
results["agree"] = np.isclose(results["gwsnr"], results["bilby"], rtol=SNR_RTOL, atol=SNR_ATOL)
audit = pd.DataFrame(all_audits)
print("All settings and inner-product decomposition checks passed.")

00:08 bilby INFO    : Waveform generator initiated with
  frequency_domain_source_model: bilby.gw.source.lal_binary_black_hole
  time_domain_source_model: None
  parameter_conversion: bilby.gw.conversion.convert_to_lal_binary_black_hole_parameters



Initializing GWSNR class...



00:08 bilby INFO    : Waveform generator initiated with
  frequency_domain_source_model: bilby.gw.source.lal_binary_black_hole
  time_domain_source_model: None
  parameter_conversion: bilby.gw.conversion.convert_to_lal_binary_black_hole_parameters



Initializing GWSNR class...





All settings and inner-product decomposition checks passed.


## 4. Results

`difference = GWSNR − Bilby`; percentages use Bilby as the denominator. `agree` uses the numerical tolerances defined above (`rtol=1e-7`, allowing for the small antenna-response difference), not a scientific acceptance threshold. The network SNR is the quadrature sum of detector SNRs.

In [7]:
with pd.option_context("display.float_format", "{:.10g}".format, "display.max_columns", None):
    display(results.set_index(["model", "detector"])[
        ["gwsnr_diagonal_only", "gwsnr", "bilby", "difference", "difference_percent", "agree"]
    ])

# Both waveform models must now agree, detector by detector and in the network.
np.testing.assert_allclose(results["gwsnr"], results["bilby"], rtol=SNR_RTOL, atol=SNR_ATOL)

for approximant in parameters_by_model:
    subset = results[results["model"] == approximant]
    network = subset[subset["detector"] == "network"].iloc[0]
    status = "AGREE" if subset["agree"].all() else "MISMATCH"
    print(f"{approximant}: {status}; network GWSNR={network['gwsnr']:.10f}, "
          f"Bilby={network['bilby']:.10f}, difference={network['difference_percent']:+.8f}%")

gwsnr_diagonal_only       gwsnr       bilby  \
model         detector                                                
IMRPhenomD    L1                40.25998635 40.25998635 40.25998664   
              H1                27.44488205 27.44488205 27.44488259   
              V1                16.15191035 16.15191035  16.1519104   
              network           51.33198086 51.33198086  51.3319814   
IMRPhenomXPHM L1                39.38053956 39.22406156 39.22406184   
              H1                26.83803239 26.73413244 26.73413297   
              V1                15.71684779 15.72297108 15.72297113   
              network           50.18093445 50.00452642 50.00452694   

                             difference  difference_percent  agree  
model         detector                                              
IMRPhenomD    L1        -2.94231782e-07    -7.308293087e-07   True  
              H1       -5.430439707e-07    -1.978671138e-06   True  
              V1       -4.397880815e-08     -2.72282393e-07   True  
              network  -5.349469561e-07    -1.042131906e-06   True  
IMRPhenomXPHM L1       -2.849138667e-07    -7.263752231e-07   True  
              H1       -5.278292434e-07    -1.974364547e-06   True  
              V1       -4.603211323e-08    -2.927698134e-07   True  
              network  -5.201588635e-07    -1.040223547e-06   True

IMRPhenomD: AGREE; network GWSNR=51.3319808639, Bilby=51.3319813988, difference=-0.00000104%
IMRPhenomXPHM: AGREE; network GWSNR=50.0045264161, Bilby=50.0045269363, difference=-0.00000104%


### Cross-term and settings diagnostics

`cross_term_rho2` is the mixed contribution $C$ evaluated with Bilby's antenna factors; it can be positive or negative. `antenna_term_rho2` is $A$, the difference between the **full** powers evaluated with Bilby's and GWSNR's antenna factors. `rho2_residual` is $\rho_{\rm Bilby}^2-\rho_{\rm GWSNR}^2-A$ and should vanish to numerical precision.

This tests a single source per model; it does not establish a population-wide error bound or isolate precession from higher-mode effects.

In [8]:
with pd.option_context("display.float_format", "{:.10g}".format, "display.max_columns", None):
    display(results.set_index(["model", "detector"])[
        ["diagonal_rho2", "cross_term_rho2", "antenna_term_rho2", "rho2_residual"]
    ])
    display(audit.drop(columns="ASD_file").set_index(["model", "detector"]))

print("Resolved ASD files:")
for detector, path in audit.drop_duplicates("detector")[["detector", "ASD_file"]].itertuples(index=False, name=None):
    print(f"{detector}: {path}")

diagonal_rho2  cross_term_rho2  antenna_term_rho2  \
model         detector                                                      
IMRPhenomD    L1          1620.866524  1.777299572e-17    2.369153026e-05   
              H1          753.2215804  8.042923128e-18     2.98075538e-05   
              V1          260.8842095 -3.063772999e-18    1.420685408e-06   
              network     2634.972314  2.275214585e-17    5.491976947e-05   
IMRPhenomXPHM L1          1550.826918     -12.29989075    2.235096281e-05   
              H1          720.2800109     -5.566145255    2.822211582e-05   
              V1          247.0193057     0.1925153902     1.44752255e-06   
              network     2518.126235     -17.67352062    5.202060117e-05   

                          rho2_residual  
model         detector                   
IMRPhenomD    L1        4.774847184e-12  
              H1        2.046363079e-12  
              V1       -1.818989404e-12  
              network    5.00222086e-12  
IMRPhenomXPHM L1       -4.774847184e-12  
              H1       -1.818989404e-12  
              V1         5.96855898e-13  
              network   -5.99698069e-12

max_PSD_relative_difference    Fp_difference  \
model         detector                                                 
IMRPhenomD    L1                                  0 -1.220878265e-08   
              H1                                  0  1.595518351e-08   
              V1                                  0  4.665581255e-10   
IMRPhenomXPHM L1                                  0 -1.220878265e-08   
              H1                                  0  1.595518351e-08   
              V1                                  0  4.665581255e-10   

                          Fc_difference  
model         detector                   
IMRPhenomD    L1         4.95037511e-09  
              H1       -2.019920953e-09  
              V1       -1.392967726e-08  
IMRPhenomXPHM L1         4.95037511e-09  
              H1       -2.019920953e-09  
              V1       -1.392967726e-08

Resolved ASD files:
L1: /Users/phurailatpamhemantakumar/anaconda3/envs/ler/lib/python3.10/site-packages/bilby/gw/detector/noise_curves/aLIGO_O4_high_asd.txt
H1: /Users/phurailatpamhemantakumar/anaconda3/envs/ler/lib/python3.10/site-packages/bilby/gw/detector/noise_curves/aLIGO_O4_high_asd.txt
V1: /Users/phurailatpamhemantakumar/anaconda3/envs/ler/lib/python3.10/site-packages/bilby/gw/detector/noise_curves/AdV_asd.txt
